# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atulpatel-net/FlyRank_ML_In/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method Choice

I chose the **Random Forest Classifier** for this task because it is well-suited for structured tabular data and can model non-linear relationships between features such as impressions, clicks, CTR, and average position. It is robust to noisy data, requires minimal preprocessing, and provides feature importance scores that help interpret the model's decisions.

Random Forest also reduces overfitting by combining predictions from multiple decision trees, making it more reliable than a single Decision Tree. This makes it a good choice for predicting whether a content page is likely to experience a significant decline in search impressions.

The model will be compared against the rule-based baseline developed in Week 04 using the same dataset and evaluation metrics to ensure an honest comparison.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split Design

I use a **grouped train-test split based on `client_hash_id`** so that all content pages belonging to the same client remain in either the training set or the testing set, but never both. This prevents the model from learning client-specific patterns during training and then being evaluated on the same client's data.

The split is also **time-aware** because only historical information available at the prediction time is used to create the features. No future-window metrics, label-derived columns, or information from later dates are included in the training data.

This provides an honest evaluation because the model is tested on unseen clients while using only information that would have been available when making a real prediction.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [2]:
import duckdb
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

In [10]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

In [11]:
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "fact_daily_sample":
        f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')"
}

In [24]:
data = con.sql(f"""

WITH bounds AS (
    SELECT MAX(report_date) AS end_d
    FROM {TABLES['fact_daily']}
),

features AS (

SELECT

    client_hash_id,
    content_hash_id,

    SUM(
        CASE
            WHEN report_date BETWEEN end_d - INTERVAL 59 DAY
                                 AND end_d - INTERVAL 30 DAY
            THEN gsc_impressions
            ELSE 0
        END
    ) AS imp_prev30,

    SUM(
        CASE
            WHEN report_date BETWEEN end_d - INTERVAL 29 DAY
                                 AND end_d
            THEN gsc_impressions
            ELSE 0
        END
    ) AS imp_last30,

    SUM(
        CASE
            WHEN report_date BETWEEN end_d - INTERVAL 59 DAY
                                 AND end_d - INTERVAL 30 DAY
            THEN gsc_clicks
            ELSE 0
        END
    ) AS clk_prev30,

    AVG(
        CASE
            WHEN report_date BETWEEN end_d - INTERVAL 59 DAY
                                 AND end_d - INTERVAL 30 DAY
            THEN gsc_avg_position
        END
    ) AS pos_prev30

FROM {TABLES['fact_daily']}, bounds

GROUP BY
    client_hash_id,
    content_hash_id

)

SELECT *

FROM features

WHERE imp_prev30 > 0

""").df()

print(data.shape)
data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(237429, 6)


,client_hash_id,content_hash_id,imp_prev30,imp_last30,clk_prev30,pos_prev30
0,client_9958f0a7ae1df715,content_d5479697a0828b33,141.0,58.0,0.0,63.372129
1,client_9958f0a7ae1df715,content_d7daac9d28863fda,127.0,179.0,0.0,20.027114
2,client_9958f0a7ae1df715,content_b313be479d6d3707,33.0,65.0,0.0,27.153646
3,client_9958f0a7ae1df715,content_5175438fecb054a4,39.0,24.0,0.0,64.730159
4,client_9958f0a7ae1df715,content_3a204f16288e29ed,5.0,5.0,0.0,67.000000


In [28]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd


# Create Target Label


data["is_declining"] = (
    data["imp_last30"] < 0.8 * data["imp_prev30"]
).astype(int)


# Features


feature_cols = [
    "imp_prev30",
    "clk_prev30",
    "pos_prev30"
]

X = data[feature_cols]
y = data["is_declining"]
groups = data["client_hash_id"]

# Honest Group Split


gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]


# Week-04 Rule-Based Baseline


baseline_test = data.iloc[test_idx].copy()

baseline_pred = (
    (
        (baseline_test["imp_prev30"] >= 100)
        &
        (baseline_test["clk_prev30"] == 0)
        &
        (baseline_test["pos_prev30"] <= 10)
    )
).astype(int)

baseline_pred = baseline_pred.values


# Random Forest


rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)


# Comparison Table

results = pd.DataFrame({

    "Model": [
        "Week-4 Rule Baseline",
        "Random Forest"
    ],

    "Accuracy": [
        accuracy_score(y_test, baseline_pred),
        accuracy_score(y_test, rf_pred)
    ],

    "Precision": [
        precision_score(y_test, baseline_pred, zero_division=0),
        precision_score(y_test, rf_pred, zero_division=0)
    ],

    "Recall": [
        recall_score(y_test, baseline_pred, zero_division=0),
        recall_score(y_test, rf_pred, zero_division=0)
    ],

    "F1 Score": [
        f1_score(y_test, baseline_pred, zero_division=0),
        f1_score(y_test, rf_pred, zero_division=0)
    ]

})

display(results.round(3))

,Model,Accuracy,Precision,Recall,F1 Score
0,Week-4 Rule Baseline,0.346,0.517,0.009,0.018
1,Random Forest,0.617,0.670,0.814,0.735


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### Errors and Interpretation

The Random Forest model outperformed the Week-4 rule-based baseline across almost all evaluation metrics. The baseline relied on fixed thresholds using impressions, clicks, and average position, making it highly specific but unable to identify many declining pages. This resulted in a very low recall and F1 score.

In contrast, the Random Forest learned more complex relationships between the available features and achieved substantially better recall and F1 score, indicating that it detected a much larger proportion of declining pages while maintaining reasonable precision.

Some prediction errors are still expected because page performance is influenced by factors that are not included in the model, such as seasonality, content updates, search intent, competition, and algorithm changes. Adding more historical and content-related features could further improve performance.

### Model vs Baseline

| Model | Accuracy | Precision | Recall | F1 Score |
|--------|---------:|----------:|-------:|---------:|
| Week-4 Rule Baseline | 0.346 | 0.517 | 0.009 | 0.018 |
| Random Forest | 0.617 | 0.670 | 0.814 | 0.735 |

The results show that the Random Forest provides a significant improvement over the transparent rule-based baseline while preserving an honest evaluation using the same data split and metrics.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.